# PLINK / GWAS QC Workflow Agent
## Colab 02 — RAG Retrieval

這個 Notebook 是期末專案第二階段。

### 本階段目標
1. 從 GitHub 下載目前的 PLINK / GWAS Wiki
2. 讀取 `wiki/*.md`
3. 進行簡單 chunking
4. 使用 multilingual sentence embedding
5. 建立 FAISS index
6. 根據研究需求 retrieve relevant Wiki chunks
7. 保留 `source_file` 與 `source_id`

> 本階段先只做 Retrieval，不呼叫 LLM。
> Groq API Key 會在下一個 Notebook 的 Planning Agent 使用。

## 0. Colab Secret

你目前的 Groq API Key 存在 Colab Secret：

```text
Groq
```

請確認左側 Secrets 面板中，這個 Secret 的「筆記本存取權」已開啟。

本 Notebook 不會印出 API Key。

In [ ]:
# 可選：只測試 Secret 是否可讀，不顯示內容
from google.colab import userdata

try:
    _groq_key = userdata.get("Groq")
    print("Groq Secret: AVAILABLE")
except Exception:
    print("Groq Secret: NOT AVAILABLE")
    print("這不影響本 Notebook 的 RAG Retrieval；下一階段再處理即可。")

## 1. 安裝套件

In [ ]:
!pip -q install sentence-transformers faiss-cpu

## 2. Clone GitHub Repository

使用目前的公開 Repository：

`b156698-commits/PLINK-GWAS-Quality-Control-Wiki`

每次重新執行時會先刪掉舊資料夾，確保抓到 GitHub 上最新版本。

In [ ]:
import os
import shutil
from pathlib import Path

REPO_URL = "https://github.com/b156698-commits/PLINK-GWAS-Quality-Control-Wiki.git"
REPO_DIR = Path("/content/PLINK-GWAS-Quality-Control-Wiki")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone -q {REPO_URL} {REPO_DIR}

print("Repository cloned.")
print("Wiki path:", REPO_DIR / "wiki")

## 3. 檢查 Wiki 檔案

In [ ]:
wiki_dir = REPO_DIR / "wiki"

wiki_files = sorted(wiki_dir.glob("*.md"))

print("Wiki files:")
for f in wiki_files:
    print("-", f.name)

print("\nTotal:", len(wiki_files))

正常應該看到：

```text
index.md
01-plink-basics.md
02-missingness.md
03-maf-hwe.md
04-ld-pruning.md
05-relatedness.md
06-pca.md
07-association.md
08-gwas-results.md
```

## 4. 讀取 Wiki 並擷取 Source ID

每個 Wiki chunk 除了文字內容，也保留：

- `source_file`
- `source_ids`

這樣之後 Planner 的每一步都可以追溯來源。

In [ ]:
import re

def extract_source_ids(text):
    # 抓 S01, S02, ...
    ids = sorted(set(re.findall(r"\bS\d{2}\b", text)))
    return ids

documents = []

for file_path in wiki_files:
    text = file_path.read_text(encoding="utf-8")
    documents.append({
        "source_file": file_path.name,
        "source_ids": extract_source_ids(text),
        "text": text
    })

for doc in documents:
    print(doc["source_file"], doc["source_ids"])

## 5. Chunking

第一版先使用簡單、透明的 paragraph-based chunking。

原則：
- 以 Markdown 空行切段
- 避免把 chunk 切得太碎
- 每個 chunk 保留來源 metadata

之後如果效果不夠，再升級 chunking。

In [ ]:
def chunk_markdown(text, max_chars=1200):
    paragraphs = [
        p.strip()
        for p in re.split(r"\n\s*\n", text)
        if p.strip()
    ]

    chunks = []
    current = ""

    for p in paragraphs:
        candidate = p if not current else current + "\n\n" + p

        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = p

    if current:
        chunks.append(current)

    return chunks

chunks = []

for doc in documents:
    file_chunks = chunk_markdown(doc["text"], max_chars=1200)

    for i, chunk_text in enumerate(file_chunks):
        chunks.append({
            "chunk_id": f'{doc["source_file"]}::chunk_{i+1}',
            "source_file": doc["source_file"],
            "source_ids": doc["source_ids"],
            "text": chunk_text
        })

print("Total chunks:", len(chunks))

for c in chunks[:5]:
    print("\n---")
    print(c["chunk_id"])
    print(c["source_ids"])
    print(c["text"][:300].replace("\n", " "))

## 6. Embedding Model

Wiki 內容同時包含中文與英文，因此使用 multilingual sentence embedding。

第一版：

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

它的優點是：
- 支援中文 / 英文
- Colab 容易執行
- 模型不大
- 適合先做 MVP

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(EMBEDDING_MODEL)

texts = [c["text"] for c in chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)

## 7. 建立 FAISS Index

In [ ]:
import faiss

dimension = embeddings.shape[1]

# cosine similarity: normalized vector + inner product
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS index size:", index.ntotal)

## 8. Retrieval Function

輸入一個問題後：

```text
Query
↓
Embedding
↓
FAISS similarity search
↓
Top-k Wiki chunks
```

回傳結果會保留：
- similarity score
- source file
- source IDs
- chunk text

In [ ]:
def retrieve(query, top_k=5):
    query_vector = model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_vector, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        item = chunks[int(idx)].copy()
        item["score"] = float(score)
        results.append(item)

    return results

def print_results(results):
    for rank, r in enumerate(results, start=1):
        print("=" * 80)
        print(f"Rank: {rank}")
        print(f"Score: {r['score']:.4f}")
        print(f"Source file: {r['source_file']}")
        print(f"Source IDs: {r['source_ids']}")
        print("-" * 80)
        print(r["text"][:900])
        print()

## 9. Retrieval Test 1 — Missingness

預期優先找到：
- `02-missingness.md`
- S02

In [ ]:
query = "GWAS 中 variant missingness 和 sample missingness 要怎麼做？PLINK 使用什麼指令？"

results = retrieve(query, top_k=5)
print_results(results)

## 10. Retrieval Test 2 — PCA / Population Structure

預期優先找到：
- `06-pca.md`
- S05

In [ ]:
query = "case-control GWAS 要如何用 PCA 處理 population structure？principal components 如何放進 association model？"

results = retrieve(query, top_k=5)
print_results(results)

## 11. Retrieval Test 3 — Association Analysis

預期優先找到：
- `07-association.md`
- S05 / S06 / S08

In [ ]:
query = "case-control GWAS 使用 PLINK logistic regression 時，phenotype file 和 covariate file 格式是什麼？"

results = retrieve(query, top_k=5)
print_results(results)

## 12. Study Information → Retrieval Query

現在把第一階段的 `study_info` 轉成 RAG query。

之後 Planner 不會自己隨便查，而是先根據研究需求 retrieve Wiki。

In [ ]:
study_info = {
    "study_design": "case-control",
    "data_format": "plink_binary",
    "sample_size": 3000,
    "phenotype": "Disease Status",
    "covariates": ["Age", "Sex"],
    "need_pca": True,
    "need_relatedness": True,
    "association": "logistic"
}

def build_retrieval_query(study_info):
    parts = [
        f"Study design: {study_info.get('study_design')}",
        f"Data format: {study_info.get('data_format')}",
        "Need GWAS quality control workflow",
        "Need variant and sample missingness",
        "Need MAF and HWE filtering",
    ]

    if study_info.get("need_relatedness"):
        parts.append("Need LD pruning and relatedness / IBD / PI_HAT")

    if study_info.get("need_pca"):
        parts.append("Need PCA and population structure adjustment")

    if study_info.get("association") == "logistic":
        parts.append("Need case-control logistic association analysis with phenotype and covariates")

    return ". ".join(parts)

study_query = build_retrieval_query(study_info)

print("Query:")
print(study_query)

study_results = retrieve(study_query, top_k=8)

print("\nRetrieved knowledge:")
print_results(study_results)

## 13. 建立 Planner 可使用的 Context

將 retrieval results 整理成一段 context。

之後會送給 Groq LLM。

In [ ]:
def build_rag_context(results):
    blocks = []

    for i, r in enumerate(results, start=1):
        source_ids = ", ".join(r["source_ids"]) if r["source_ids"] else "N/A"

        block = f'''
[Retrieved Chunk {i}]
Source file: {r["source_file"]}
Source IDs: {source_ids}

{r["text"]}
'''.strip()

        blocks.append(block)

    return "\n\n".join(blocks)

rag_context = build_rag_context(study_results)

print(rag_context[:5000])

## 14. Retrieval Coverage Check

第一版做簡單檢查，確認 study requirement 對應的 Wiki 主題是否有被 retrieve。

這不是最終 Validator，只是 Retrieval node 自己的基本 coverage check。

In [ ]:
def retrieval_coverage_check(study_info, results):
    retrieved_files = {r["source_file"] for r in results}
    errors = []

    expected = {
        "02-missingness.md": True,
        "03-maf-hwe.md": True,
        "04-ld-pruning.md": study_info.get("need_relatedness") or study_info.get("need_pca"),
        "05-relatedness.md": study_info.get("need_relatedness"),
        "06-pca.md": study_info.get("need_pca"),
        "07-association.md": True
    }

    for file_name, needed in expected.items():
        if needed and file_name not in retrieved_files:
            errors.append(f"Relevant Wiki chapter not retrieved: {file_name}")

    return errors

coverage_errors = retrieval_coverage_check(
    study_info,
    study_results
)

if coverage_errors:
    print("RETRIEVAL COVERAGE: REVIEW")
    for e in coverage_errors:
        print("-", e)
else:
    print("RETRIEVAL COVERAGE: PASS")

## 15. 第二階段完成標準

這個 Notebook 跑完後，應該確認：

### Test 1
Missingness query 能找到 `02-missingness.md`

### Test 2
PCA query 能找到 `06-pca.md`

### Test 3
Association query 能找到 `07-association.md`

### Study Query
能一次 retrieve 多個與研究需求相關的 Wiki chapters。

最後：

```text
RETRIEVAL COVERAGE: PASS
```

若不是 PASS，也先不要硬改 Planner，而是先調整：
- query
- top_k
- chunking

---

# 下一階段

下一個 Notebook：

**`03_planning_agent_with_groq.ipynb`**

將正式使用 Colab Secret：

```python
from google.colab import userdata
GROQ_API_KEY = userdata.get("Groq")
```

串接：

```text
Study Information
↓
RAG Retrieval
↓
Groq Planning Agent
↓
QC Plan JSON
↓
Plan Validator
↓
FAIL → LLM Revise → Recheck
↓
PASS
```

這時才會第一次真正用到你的 Groq API Key。